In [2]:
# 1. Deterministic Guardrails
import re
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads.",
]

print("===Deterministic Guardrail Demo===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

===Deterministic Guardrail Demo===
BLOCKED: How do I hack into a database?
ALLOWED: What is the capital of France?
BLOCKED: Explain how malware spreads.


In [3]:
# 2. Model-Based Guardrails
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.7
)



def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""

    prompt = f"""Is the following user input sae to process? Reply with only 'SAFE' or 'UNSAFE'. Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()


print("===Deterministic Guardrail Demo===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

===Deterministic Guardrail Demo===
UNSAFE: How do I hack into a database?
SAFE: What is the capital of France?
SAFE: Explain how malware spreads.


In [4]:
# 3. PII middleware
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool 

# define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# create agent with PII middleware
agent = create_agent(
    model = "google_genai:gemini-2.5-flash",
    tools = [customer_lookup],
    middleware=[
        # redact emails in user input
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = True,
        ),
        # mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_input = True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
                    "api_key",
                    detector = r"sk-[a-zA-z0-9]{32}",
                    strategy = "block",
                    apply_to_input = True,
                ),
    ]
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [5]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
[{'type': 'text', 'text': 'I found your record. How can I help you?', 'extras': {'signature': 'CpUCARFNMg+Xm8IYWytFkm/J5BcAYXmxcAQBcZsDGBbx38b/BTQTKDooAP7ChZjbficWB7yUnn1SjDrm5pKETu0IOwOsIYGQJ7hmwSn2oxi/FWzbbwv9uPEA4Hy1lG2oICxwzzQNMMZ91HZFAfc8N942DVgH6xDC5JRWHKBZgpuIjjb8TFIh/ahVoxJDyxR2bfkWUxFdvz441AYtVRf23IgzjXHeZPsllFGXMs9XP88o+Rp5zn5JOa9yDdX1pgQn6xBL8TNkL7fkB9b/UEr8Ky5b7nEa7UR717DZQ7RhCE+uCLHTiOhX8n7Fbz+zGT77r5+M+tzNfjeUGnJiWcH76WGJsj1LOadGJ/EiWW6xpiLV+7ppg44w6w=='}}]


In [6]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='410a8e64-b87a-4b09-8919-c015510d4d75'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'customer_lookup', 'arguments': '{"query": "[REDACTED_EMAIL] ****-****-****-5100"}'}, '__gemini_function_call_thought_signatures__': {'65da213f-0faa-4165-bf73-afe5a7cb0d3f': 'CsgDARFNMg+QEG4Fo8XYGD/NTEtuJnpg+IjiNC/ysFpZ+YXf2KG+Ff4Cbn4PhU6jlooQCP8YD1u5rE33EOyw5kLQqeSyZ5R9I2rBmEXt5eNYeB9G67+4+Djm+hLE/OPA3k+eyvkas8wFjGgl/usV4Zu4nyLbvCOVXnzTQR+muLxjvfn1//swkqF+oLFZaLggZJojkCwxQUcxk34gExaI7+ecpFGCRNcl957SxC/kdVBeCt/mjuutZdMlU9WXQeCLQ98JZEf+I8SQrJE2q4k/ERg82phwXtPjxWkH9s5YrRplArbmgiVpJO3iGAlOBS1wQKitg1hrSlgUVbfd8brAeFGRz3mpn9pu/Cc8SMXG4O2tKjGaWY93LrkiPbnaTU4wjmgkZrIIqcApSuOrhnd3ulWodD43cpt5sFomn0POC3KH3OADpUYyGRsootJ5eVsaw6x85kgQi3OkLZQ5FLecqsjd0nmtl2AHiQ30BVg7Q7rm8Y62YnZTD7Js5FFpTSwNBXhZsB4MVOn0dBKChINCoDoo2wbEj/RDV2pB

In [7]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [8]:
# 4. HITL middleware
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"



# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [9]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='49a037f7-53b2-4c83-a1ad-4ba02932cdb7'), AIMessage(content=[{'type': 'text', 'text': 'I can do that. What would you like the email body to say?', 'extras': {'signature': 'CpsCARFNMg/1vsCpGVXOdEMpg4rF+sebeO6+97wE1pmUP5+CDvdC+f5k0Q4h3s29aylQgRQAsrm0fH2hlovrg6Wyguo/EVCJwbxE5w2Q0iSemf7K2YoOQiiUiD0xLsIGFjL2QzwoOEB1DFt7nWte2muyAjI41ff5O0FxFORPKMt6XZpUUSj6+OT4Ag2fvMZgzkBQB524Pls1Y01t1g9h4/QsUObIrHYjVSG1bgWNeCYafsp3qzBj00ayWOP/vikUh/2+7lMQ9yRk5aWLfpYiywkNVIukLCTislFfpGcgjogF+Tijqx2lQhC1lrxZwE1WBCscThcvAX03nsz61+3pmzB0nh/k+7deQfqrTph+jzRg0KwXyoMFVhdFnH4B9w=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ffa8c-04cf-7843-847a-e347544b21ae-0', tool_calls=[], invalid_tool_calls=[], usag

In [10]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
[{'type': 'text', 'text': 'I can do that. What would you like the email body to say?', 'extras': {'signature': 'CpsCARFNMg/1vsCpGVXOdEMpg4rF+sebeO6+97wE1pmUP5+CDvdC+f5k0Q4h3s29aylQgRQAsrm0fH2hlovrg6Wyguo/EVCJwbxE5w2Q0iSemf7K2YoOQiiUiD0xLsIGFjL2QzwoOEB1DFt7nWte2muyAjI41ff5O0FxFORPKMt6XZpUUSj6+OT4Ag2fvMZgzkBQB524Pls1Y01t1g9h4/QsUObIrHYjVSG1bgWNeCYafsp3qzBj00ayWOP/vikUh/2+7lMQ9yRk5aWLfpYiywkNVIukLCTislFfpGcgjogF+Tijqx2lQhC1lrxZwE1WBCscThcvAX03nsz61+3pmzB0nh/k+7deQfqrTph+jzRg0KwXyoMFVhdFnH4B9w=='}}]


In [11]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='49a037f7-53b2-4c83-a1ad-4ba02932cdb7'), AIMessage(content=[{'type': 'text', 'text': 'I can do that. What would you like the email body to say?', 'extras': {'signature': 'CpsCARFNMg/1vsCpGVXOdEMpg4rF+sebeO6+97wE1pmUP5+CDvdC+f5k0Q4h3s29aylQgRQAsrm0fH2hlovrg6Wyguo/EVCJwbxE5w2Q0iSemf7K2YoOQiiUiD0xLsIGFjL2QzwoOEB1DFt7nWte2muyAjI41ff5O0FxFORPKMt6XZpUUSj6+OT4Ag2fvMZgzkBQB524Pls1Y01t1g9h4/QsUObIrHYjVSG1bgWNeCYafsp3qzBj00ayWOP/vikUh/2+7lMQ9yRk5aWLfpYiywkNVIukLCTislFfpGcgjogF+Tijqx2lQhC1lrxZwE1WBCscThcvAX03nsz61+3pmzB0nh/k+7deQfqrTph+jzRg0KwXyoMFVhdFnH4B9w=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ffa8c-04cf-7843-847a-e347544b21ae-0', tool_calls=[], invalid_tool_calls=[], usag

In [12]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I am sorry, I cannot fulfill this request. The user has rejected the tool call.


In [15]:
# 5. Custom Guardrail (Input filter)
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool



class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"



# Create agent with content filter
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [16]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
[{'type': 'text', 'text': 'Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention. It involves the development of algorithms that can analyze and interpret data, and then use that understanding to make predictions or take actions. This learning process allows machines to improve their performance over time without being explicitly programmed for every task.', 'extras': {'signature': 'CqcFARFNMg+ZINv99yJ5GDHJ2V+fJL5MTvbx+6ZYByJN7/sSkvvUJiVZPIUuovVZbAxSGGR9cdfZBGeJXyDYYs7brMOQ4uuQv+vsfh4ufT6bk3yC3+3vYfujV4V428gZXDxEa0pZsJfiYal0YkYFjsDeJl/q0YYTE1U6TKLpMOmzIExty2gi34ws2gMKiDCQXGlvATCU7Pu4hpJKHzL9IJZsocgx1UF7JMjnbv3xjyteyQLBbQyMSBY16DD3hLwGab1czHr61SDWVN0aX3CbyA+ZsasqkjwnFgp5Brci1tuvRPk+HdZe2HBPOW+bSzOyxUQAENk7Bh3HSTZhYdFkMXsc6TPSCdFi6ziWFKqsG3OWdvA5+VrD7PAeUwrEDvHo9crqfFhFTJVbIlypi6R3V0EqU+uhaAHYSOdwAIYRGaC74wCEBH9yzmS7IjXnHd7JRnsmGI4VB50aw70NaW9TPXs

In [17]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


In [18]:
# 6. Custom Guardrail (Output filter)
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool



class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    




@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [20]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "how to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
[{'type': 'text', 'text': 'I cannot provide information or assistance on how to make a bomb. My purpose is to be helpful and harmless, and that includes refusing to generate responses that promote dangerous activities.', 'extras': {'signature': 'Cr8CARFNMg+uWRORV87qh542znFWplmQDrsDzxfOCwTXBs49xsT5fDuXfTtYOty9FEDn7ZW1W4tkJFsZ04I2Qb1i1JX8/Qt9WkSSYlR5GxpDErJmmk+5FMmFSBWW2snXkxbi1y5J0vPDpTdTzkhKyytLg6fMTvbuoJRmJPwUuLX/kocWAH3Ao9/xqA4l6QFBDWZO18o0hQnJ+R3ZL8OPlTROCe7wSbomC7zDDUaWuVGePmlY94h/p0Hk/lxRWmOr7orgQsfarVVXhESteNUrmPX+2WnrDRYM0zzZL0QB0UkcyS0MG2FjztEvyrn3a2teZ2af84vYNOBgIWUOvBDFwDyfV+vtcJuuS+O6h9RYhPpybLd/+HN6bnZOpn+A9wEdPwedCI4MBOrVzvGWYjW3rMUl8J+ZPNZvJc0SmUdp7B0c7Q=='}}]


In [21]:
# 7. Layered / Combined Guardrails
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"


# Full layered guardrail stack
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input

        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


In [22]:
config = {"configurable": {"thread_id": "test_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)


print(result["messages"][-1].content)

[{'type': 'text', 'text': 'I cannot provide information or instructions on how to make a bomb. My purpose is to be helpful and harmless, and that includes refusing to generate content that is dangerous, illegal, or promotes violence. If you are going through a difficult time, please consider reaching out to a crisis hotline or mental health professional.', 'extras': {'signature': 'Cs4JARFNMg/KXQhIzi0I4nfVUI0PJXWBsTouW3cAyTV0DmlbqcwYEwN1Oh+RHd1I3wAuO+2jH1rDUwEpPvLDOJ6b6BkgFrwqHLfztY1ofDTUk3pZiWOUcaWKTcKrMbrYezrdRzGdvMn37tIN9XYK4dn+WEnooa4X4ktbQy3PYoof1vVeAM/0xj9iE7O/CsNaYTJiPLrE1JOYWoD+wHr+lvbwurjdTomcQfflv5cJ7e7+GQ4vhooneJr9kqLZZlCUelU2w13DUL8JurBNgt6CzCYNUudgxUPEZcHpvyEPNEH+WvFfHOKH72oMZ+Qja5s7Z1rQj8XhBXjbpx/YsnN6y5lP1fI05BeQG82oSOqzQ8cDw5UvlrZpuDzHH7LY3A767odSuZriv7l62vSGbFRdFMLmaR851FjVS4JahPLQXqY3sI2fya/kqzgLlPW/FG+3X2MgqNZzWMtqz+xLgHBfne8GhyocvrjEeDNd+DPIZGNMafXX+eSZqC6Z6h0nhuR++JUWPyJ5IAdrL93A/LFQkmxNzhMQsUGKz2HiRJWFFasDrN6nuezwAVELzADcQP5Rrsg3qQA1bjaAQ3s9W9DJEJXU9gFLy+n/kY/okiYV

In [ ]:
# 8. Real-World Use Case — Healthcare Chatbot
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage


# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    



# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
    


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."




# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [25]:
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

Type 2 Diabetes can present with various symptoms. Some common ones include increased thirst, frequent urination, increased hunger, fatigue, blurred vision, slow-healing sores, and frequent infections. However, it's important to remember that these symptoms can also be associated with other conditions.

I'm an AI assistant, and while I can provide information, I cannot diagnose. It's crucial to consult a doctor for a proper diagnosis and treatment plan if you are experiencing any of these symptoms. They will be able to provide you with the most accurate information and care.

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


In [30]:
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
# Handle case where content might be a list (tool calls) or string
last_msg = approved["messages"][-1]
if isinstance(last_msg.content, str):
    print(last_msg.content)
else:
    print(f"Tool calls or structured content: {last_msg.content}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.795496765s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT